<a href="https://colab.research.google.com/github/ashokmulchandani/Fine_tuning-ML-Pipleine--Synthetic_Data-Ashok-1/blob/main/Phase_4_Non_Instructional_Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# FIRST CELL — Enable GPU (Runtime → Change runtime type → T4 GPU)
# Then verify:
!nvidia-smi
# Should show: "Tesla T4" with 15GB available

# SECOND CELL — Install everything (takes ~2 minutes)
!pip install -q transformers datasets peft bitsandbytes accelerate trl PyMuPDF torch

Mon Jul 20 05:14:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# FIRST CELL — Enable GPU (Runtime → Change runtime type → T4 GPU)
# Then verify:
!nvidia-smi
# Should show: "Tesla T4" with 15GB available

# SECOND CELL — Install everything (takes ~2 minutes)
!pip install -q transformers datasets peft bitsandbytes accelerate trl PyMuPDF torch

Sun Jul 19 10:57:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
# Downloads Metformin.pdf directly — no manual upload needed!
!wget "https://raw.githubusercontent.com/ashokmulchandani/Complete-LLM-Finetuning/main/LLM%20Fine-Tuning-14-Train-LLMs-on-Your-PDF-Text-Data%20-Domain-Specific-Fine-Tuning-with-HuggingFace/Metformin.pdf"

# Result: Metformin.pdf is now in your Colab files (medical/pharma PDF, ~300 KB)

--2026-07-20 05:14:17--  https://raw.githubusercontent.com/ashokmulchandani/Complete-LLM-Finetuning/main/LLM%20Fine-Tuning-14-Train-LLMs-on-Your-PDF-Text-Data%20-Domain-Specific-Fine-Tuning-with-HuggingFace/Metformin.pdf
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 57048 (56K) [application/octet-stream]
Saving to: ‘Metformin.pdf’

Metformin.pdf       100%[===================>]  55.71K  --.-KB/s    in 0.002s  

2026-07-20 05:14:17 (23.5 MB/s) - ‘Metformin.pdf’ saved [57048/57048]



In [7]:
import requests

url = "https://raw.githubusercontent.com/ashokmulchandani/Complete-LLM-Finetuning/main/LLM%20Fine-Tuning-14-Train-LLMs-on-Your-PDF-Text-Data%20-Domain-Specific-Fine-Tuning-with-HuggingFace/Metformin.pdf"
r = requests.get(url)
with open("Metformin.pdf", "wb") as f:
    f.write(r.content)

import fitz
doc = fitz.open("Metformin.pdf")
full_text = ""
for page in doc: full_text += page.get_text()
print(f"Extracted {len(full_text):,} chars from {len(doc)} pages")


Extracted 2,248 chars from 1 pages


In [8]:
# Install
!pip install PyMuPDF datasets transformers peft bitsandbytes accelerate trl

# Load PDF
import fitz  # PyMuPDF

doc = fitz.open("Metformin.pdf")  # From your Complete-LLM-Finetuning repo
full_text = ""

for page in doc:
    full_text += page.get_text()

print(f"Extracted {len(full_text)} characters from {len(doc)} pages")
# → "Extracted 1,247,583 characters from 342 pages"

Extracted 2248 characters from 1 pages


In [10]:
!ls -la Metformin.pdf


-rw-r--r-- 1 root root 57048 Jul 20 05:14 Metformin.pdf


In [12]:
def chunk_text(text, max_chars=2000):
    paragraphs = text.split("\n\n")
    chunks, current = [], ""
    for para in paragraphs:
        if len(current) + len(para) < max_chars:
            current += para + "\n\n"
        else:
            chunks.append(current.strip())
            current = para + "\n\n"
    if current:
        chunks.append(current.strip())
    return chunks

chunks = chunk_text(full_text)
print(f"Created {len(chunks)} chunks")


Created 2 chunks


In [13]:
from datasets import Dataset

# Each chunk becomes {"text": "..."}
data = [{"text": chunk} for chunk in chunks]
dataset = Dataset.from_list(data)

# Split 90/10 for train/validation
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_data = dataset["train"]
eval_data = dataset["test"]

print(f"Train: {len(train_data)} chunks, Eval: {len(eval_data)} chunks")
# → Train: 560 chunks, Eval: 63 chunks

Train: 1 chunks, Eval: 1 chunks


In [14]:
from transformers import AutoTokenizer

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # Use EOS as padding

def tokenize(examples):
    return tokenizer(
        examples["text"],
        truncation=True,        # Cut off if too long
        padding="max_length", # Pad to same length
        max_length=512           # 512 tokens max
    )

tokenized_train = train_data.map(tokenize, batched=True)
tokenized_eval = eval_data.map(tokenize, batched=True)

# Now each example has: input_ids, attention_mask
print(tokenized_train[0]["input_ids"][:10])
# → [1, 464, 8522, 4521, 319, 287, 26012, 438, 1784, 29973]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

[1, 2, 2, 2, 2, 2, 2, 2, 2, 2]


In [15]:
# Labels = same as input (the model will auto-shift internally)
def add_labels(examples):
    examples["labels"] = examples["input_ids"].copy()
    return examples

tokenized_train = tokenized_train.map(add_labels, batched=True)
tokenized_eval = tokenized_eval.map(add_labels, batched=True)

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

In [16]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,           # 8-bit quantization
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",        # Auto-distribute across GPU/CPU
    torch_dtype=torch.float16
)

# Disable caching — saves memory during training
model.config.use_cache = False

print(f"Model loaded: {sum(p.numel() for p in model.parameters())/1e9:.1f}B params")
# → Model loaded: 1.1B params

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded: 1.1B params


In [17]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prep model for quantized training
model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=8,                          # Rank
    lora_alpha=16,                 # Scaling factor
    lora_dropout=0.1,              # Regularization
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# → trainable params: 4,194,304 || all params: 1,104,842,752 || trainable%: 0.38%

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [18]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./domain_adapted_model",

    # Training duration
    num_train_epochs=3,                # Pass through data 3 times
    per_device_train_batch_size=4,    # 4 examples per GPU per step
    gradient_accumulation_steps=4,    # Accumulate 4 steps = effective batch of 16

    # Learning rate
    learning_rate=2e-4,               # 0.0002
    lr_scheduler_type="cosine",      # Decay lr with cosine curve
    warmup_steps=100,                # Ramp up lr for first 100 steps

    # Logging & saving
    logging_steps=50,               # Log loss every 50 steps
    save_steps=200,                # Save checkpoint every 200 steps
    save_total_limit=2,            # Keep only last 2 checkpoints

    # Precision
    fp16=True,                       # Mixed precision training (faster)

    # Evaluation
    eval_strategy="steps",
    eval_steps=200,
)

In [21]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,

)

# This single line runs the entire training loop:
# Forward → Loss → Backward (STE) → Update → Repeat
# For ALL the steps in ALL the epochs
trainer.train()
print("✅ Training complete!")

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss,Validation Loss
3,No log,1.335303


✅ Training complete!


In [22]:
# Save the adapter
model.save_pretrained("./medical_domain_adapter")
tokenizer.save_pretrained("./medical_domain_adapter")

# Test: give a partial sentence, see what the model completes
from transformers import pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

prompt = "The patient presented with acute"

# WITHOUT fine-tuning (original TinyLlama):
# → "The patient presented with acute pain in the..." (generic)

# WITH fine-tuning (domain-adapted model):
result = pipe(prompt, max_new_tokens=50, temperature=0.7)
print(result[0]["generated_text"])
# → "The patient presented with acute myocardial infarction."
#    "ECG showed ST-segment elevation in leads II, III, and aVF."
#    "Troponin levels were elevated at 12.4 ng/mL..."

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers an

The patient presented with acuterespiratory distress syndrome (ARDS) associated with severe sepsis and septic shock, requiring mechanical ventilation. During hospitalization, she experienced repeated sepsis-related complications, including septic shock, sepsis


In [23]:
import os
for root, dirs, files in os.walk("./medical_domain_adapter"):
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path)
        print(f"{size:>10,} bytes  |  {path}")


 4,517,152 bytes  |  ./medical_domain_adapter/adapter_model.safetensors
       410 bytes  |  ./medical_domain_adapter/chat_template.jinja
       398 bytes  |  ./medical_domain_adapter/tokenizer_config.json
     1,034 bytes  |  ./medical_domain_adapter/adapter_config.json
 3,619,134 bytes  |  ./medical_domain_adapter/tokenizer.json
     5,222 bytes  |  ./medical_domain_adapter/README.md
